# Tree-Based Models — Sarcasm Detection

This notebook trains and compares Random Forest and XGBoost on the SARC dataset,
using TF-IDF features augmented with engineered features (subreddit target encoding, punctuation features).

**Pipeline:**
1. Load preprocessed data
2. Train/val/test split (90/10 stratified, seed=42)
3. Feature engineering: TF-IDF + subreddit target encoding + punctuation features
4. Train Random Forest and XGBoost
5. Hyperparameter tuning
6. Evaluate best models on test set
7. Feature importance analysis
8. Comparison with logistic regression baseline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import xgboost as xgb
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

from joblib import dump, load

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

SEED = 42
np.random.seed(SEED)

Imports OK


In [2]:
train_df = pd.read_csv('../data/train_cleaned.csv')
test_df  = pd.read_csv('../data/test_cleaned.csv')

print(f'Train: {train_df.shape}, Test: {test_df.shape}')

Train: (1125667, 11), Test: (283332, 9)


In [3]:
VAL_RATIO = 0.10

train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=VAL_RATIO,
    stratify=train_df['label'],
    random_state=SEED,
)

train_subset = train_df.iloc[train_idx].reset_index(drop=True)
val_subset   = train_df.iloc[val_idx].reset_index(drop=True)

y_train = train_subset['label'].values
y_val   = val_subset['label'].values
y_test  = test_df['label'].values

print(f'Train: {len(train_subset):,}  |  Val: {len(val_subset):,}  |  Test: {len(test_df):,}')
print(f'Train label dist: {train_subset["label"].value_counts().to_dict()}')

Train: 1,013,100  |  Val: 112,567  |  Test: 283,332
Train label dist: {1: 506552, 0: 506548}


---
## 3. Feature Engineering

We build three types of features and concatenate them into a single feature matrix:

### 3a. TF-IDF (same as logistic regression baseline)
Bag-of-words with unigrams + bigrams, 50K max features.

### 3b. Subreddit Target Encoding
Instead of one-hot encoding (which would add thousands of sparse columns), we encode each subreddit
as its **sarcasm rate** in the training set. This is a single continuous feature that captures how
sarcasm-prone each community is. We add Laplace smoothing to handle rare subreddits.

### 3c. Punctuation Features
Hand-crafted features that capture stylistic sarcasm signals:
- Exclamation mark count
- Question mark count
- Ellipsis count (...)
- ALL CAPS word ratio (e.g. "GREAT idea")

In [4]:
# ── 3a. TF-IDF ──
TEXT_COL = 'text_normalized'

train_texts = train_subset[TEXT_COL].fillna('').astype(str)
val_texts   = val_subset[TEXT_COL].fillna('').astype(str)
test_texts  = test_df[TEXT_COL].fillna('').astype(str)

print('Fitting TF-IDF...')
tfidf = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=5,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train_tfidf = tfidf.fit_transform(train_texts)
X_val_tfidf   = tfidf.transform(val_texts)
X_test_tfidf  = tfidf.transform(test_texts)

print(f'TF-IDF shape: {X_train_tfidf.shape}')

Fitting TF-IDF...
TF-IDF shape: (1013100, 50000)


In [ ]:
# ── 3b. Subreddit Target Encoding ──
# Compute sarcasm rate per subreddit on training set with Laplace smoothing

SMOOTHING_ALPHA = 10

sub_stats = train_subset.groupby('subreddit')['label'].agg(['sum', 'count'])
sub_stats.columns = ['n_sarcastic', 'n_total']
sub_stats['sarcasm_rate'] = (
    (sub_stats['n_sarcastic'] + SMOOTHING_ALPHA) / 
    (sub_stats['n_total'] + 2 * SMOOTHING_ALPHA)
)

# Global fallback for unseen subreddits
global_rate = train_subset['label'].mean()
print(f'Global sarcasm rate: {global_rate:.4f}')
print(f'Subreddits in training set: {len(sub_stats):,}')
print(f'\nTop 10 most sarcastic subreddits:')
print(sub_stats[sub_stats['n_total'] >= 100].nlargest(10, 'sarcasm_rate')[['n_total', 'sarcasm_rate']])

def encode_subreddit(series, stats, global_rate):
    """Map subreddit names to their sarcasm rate."""
    return series.map(stats['sarcasm_rate']).fillna(global_rate).values.reshape(-1, 1).astype(np.float32)

X_train_sub = encode_subreddit(train_subset['subreddit'], sub_stats, global_rate)
X_val_sub   = encode_subreddit(val_subset['subreddit'], sub_stats, global_rate)
X_test_sub  = encode_subreddit(test_df['subreddit'], sub_stats, global_rate)

print(f'\nSubreddit feature shape: {X_train_sub.shape}')

Global sarcasm rate: 0.5000
Subreddits in training set: 15,638

Top 10 most sarcastic subreddits:
                      n_total  sarcasm_rate
subreddit                                  
creepyPMs                5103      0.771228
badwomensanatomy          208      0.732456
progun                    443      0.730022
Xenoblade_Chronicles      162      0.719780
Liberal                   103      0.699187
MensRights               2971      0.692076
GunsAreCool               272      0.691781
economy                   210      0.691304
environment               325      0.689855
rage                      806      0.687651

Subreddit feature shape: (1013100, 1)


In [20]:
# ── 3c. Engineered Features ──
RAW_COL = 'comment' if 'comment' in train_subset.columns else 'comment_clean'
if RAW_COL not in train_subset.columns:
    RAW_COL = TEXT_COL
print(f'Using column for engineered features: {RAW_COL}')

def extract_engineered_features(df, col):
    texts = df[col].fillna('').astype(str)

    # Punctuation
    excl_count     = texts.str.count('!').values
    ques_count     = texts.str.count(r'\?').values
    ellipsis_count = texts.str.count(r'\.{3,}').values

    # ALL CAPS ratio
    def caps_ratio(text):
        words = text.split()
        if not words: return 0.0
        return sum(1 for w in words if w.isupper() and len(w) >= 2) / len(words)
    caps = texts.apply(caps_ratio).values

    # Comment length
    word_count = texts.str.split().str.len().fillna(0).values
    char_count = texts.str.len().values

    features = np.column_stack([
        excl_count, ques_count, ellipsis_count, caps,
        word_count, char_count
    ]).astype(np.float32)
    return features

X_train_eng = extract_engineered_features(train_subset, RAW_COL)
X_val_eng   = extract_engineered_features(val_subset,   RAW_COL)
X_test_eng  = extract_engineered_features(test_df,      RAW_COL)

eng_feature_names = [
    'excl_count', 'ques_count', 'ellipsis_count', 'caps_ratio',
    'word_count', 'char_count'
]
print(f'Engineered features shape: {X_train_eng.shape}')
print(pd.DataFrame(X_train_eng[:5], columns=eng_feature_names))

Using column for engineered features: comment
Engineered features shape: (1013100, 6)
   excl_count  ques_count  ellipsis_count  caps_ratio  word_count  char_count
0         0.0         0.0             0.0        0.00         2.0        15.0
1         0.0         0.0             0.0        0.00        14.0        65.0
2         0.0         0.0             0.0        0.25         4.0        16.0
3         0.0         0.0             0.0        0.00         7.0        36.0
4         1.0         0.0             0.0        0.00         6.0        29.0


In [7]:
# ── Combine features ──
# Version A: raw sparse (same as before, for RF which handles sparse ok)
X_train_sparse = hstack([X_train_tfidf, csr_matrix(X_train_sub), csr_matrix(X_train_eng)])
X_val_sparse   = hstack([X_val_tfidf,   csr_matrix(X_val_sub),   csr_matrix(X_val_eng)])
X_test_sparse  = hstack([X_test_tfidf,  csr_matrix(X_test_sub),  csr_matrix(X_test_eng)])

# Version B: SVD-reduced dense (for XGBoost)
# Reduces 50k sparse TF-IDF dims to 300 dense dims so trees can split meaningfully
print('Fitting TruncatedSVD (n=300)...')
svd = TruncatedSVD(n_components=300, random_state=SEED)
normalizer = Normalizer()

X_train_svd = normalizer.fit_transform(svd.fit_transform(X_train_tfidf))
X_val_svd   = normalizer.transform(svd.transform(X_val_tfidf))
X_test_svd  = normalizer.transform(svd.transform(X_test_tfidf))

X_train_dense = np.hstack([X_train_svd, X_train_sub, X_train_eng])
X_val_dense   = np.hstack([X_val_svd,   X_val_sub,   X_val_eng])
X_test_dense  = np.hstack([X_test_svd,  X_test_sub,  X_test_eng])

print(f'Sparse feature matrix: {X_train_sparse.shape}')
print(f'Dense (SVD) feature matrix: {X_train_dense.shape}')

Fitting TruncatedSVD (n=300)...
Sparse feature matrix: (1013100, 50008)
Dense (SVD) feature matrix: (1013100, 308)


In [22]:
def evaluate(model, X, y_true, set_name='Test'):
    """Predict, print metrics, return predictions and probabilities."""
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    
    print(f'\n=== {set_name} Results ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1 Score:  {f1:.4f}')
    print(f'ROC AUC:   {auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['Not Sarcastic', 'Sarcastic']))
    
    return y_pred, y_prob

In [23]:
rf_configs = [
    {'n_estimators': 100, 'max_depth': 50},
    {'n_estimators': 200, 'max_depth': 50},
    {'n_estimators': 200, 'max_depth': None},
    {'n_estimators': 300, 'max_depth': 50},
    {'n_estimators': 300, 'max_depth': None},
]

rf_results = []
for cfg in rf_configs:
    label = f'n={cfg["n_estimators"]}, depth={cfg["max_depth"]}'
    print(f'RF: {label}...', end=' ')
    t0 = time.time()
    
    model = RandomForestClassifier(
        n_estimators=cfg['n_estimators'],
        max_depth=cfg['max_depth'],
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_train_sparse, y_train)
    
    val_pred = model.predict(X_val_sparse)
    val_prob = model.predict_proba(X_val_sparse)[:, 1]
    
    rf_results.append({
        'config': label,
        'val_acc': accuracy_score(y_val, val_pred),
        'val_f1': f1_score(y_val, val_pred),
        'val_auc': roc_auc_score(y_val, val_prob),
        'time_s': time.time() - t0,
    })
    print(f'val_f1={rf_results[-1]["val_f1"]:.4f}, val_auc={rf_results[-1]["val_auc"]:.4f} ({rf_results[-1]["time_s"]:.1f}s)')

rf_results_df = pd.DataFrame(rf_results)
print('\n')
print(rf_results_df.to_string(index=False))

RF: n=100, depth=50... val_f1=0.6627, val_auc=0.7542 (20.4s)
RF: n=200, depth=50... val_f1=0.6654, val_auc=0.7560 (32.5s)
RF: n=200, depth=None... val_f1=0.7046, val_auc=0.7848 (215.4s)
RF: n=300, depth=50... val_f1=0.6663, val_auc=0.7565 (48.6s)
RF: n=300, depth=None... val_f1=0.7052, val_auc=0.7852 (323.6s)


           config  val_acc   val_f1  val_auc     time_s
  n=100, depth=50 0.688976 0.662697 0.754221  20.413131
  n=200, depth=50 0.690469 0.665373 0.755969  32.547265
n=200, depth=None 0.713726 0.704594 0.784790 215.432923
  n=300, depth=50 0.690993 0.666315 0.756487  48.624522
n=300, depth=None 0.714339 0.705229 0.785164 323.622450


In [ ]:
# XGBoost on SVD-reduced dense features
# We sweep over key hyperparameters since the dense feature space


xgb_configs = [
    {'max_depth': 4, 'learning_rate': 0.1,  'colsample_bytree': 0.7},
    {'max_depth': 6, 'learning_rate': 0.1,  'colsample_bytree': 0.7},
    {'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.7},
    {'max_depth': 6, 'learning_rate': 0.1,  'colsample_bytree': 0.9},
    {'max_depth': 8, 'learning_rate': 0.1,  'colsample_bytree': 0.7},
]

xgb_results = []
for cfg in xgb_configs:
    label = f"depth={cfg['max_depth']}, lr={cfg['learning_rate']}, col={cfg['colsample_bytree']}"
    print(f'XGB: {label}...', end=' ')
    t0 = time.time()

    model = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=cfg['max_depth'],
        learning_rate=cfg['learning_rate'],
        colsample_bytree=cfg['colsample_bytree'],
        subsample=0.8,
        min_child_weight=5,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        eval_metric='logloss',
        early_stopping_rounds=10,
        verbosity=0,
    )
    model.fit(X_train_dense, y_train,
              eval_set=[(X_val_dense, y_val)],
              verbose=False)

    val_pred = model.predict(X_val_dense)
    val_prob = model.predict_proba(X_val_dense)[:, 1]
    xgb_results.append({
        'config': label,
        'best_iter': model.best_iteration,
        'val_acc': accuracy_score(y_val, val_pred),
        'val_f1': f1_score(y_val, val_pred),
        'val_auc': roc_auc_score(y_val, val_prob),
        'time_s': time.time() - t0,
    })
    print(f"val_f1={xgb_results[-1]['val_f1']:.4f}, "
          f"val_auc={xgb_results[-1]['val_auc']:.4f}, "
          f"best_iter={model.best_iteration} ({xgb_results[-1]['time_s']:.1f}s)")

xgb_results_df = pd.DataFrame(xgb_results)
print('\n', xgb_results_df.to_string(index=False))

XGB: depth=4, lr=0.1, col=0.7... val_f1=0.6879, val_auc=0.7710, best_iter=999 (93.0s)
XGB: depth=6, lr=0.1, col=0.7... val_f1=0.6970, val_auc=0.7809, best_iter=999 (132.3s)
XGB: depth=6, lr=0.05, col=0.7... val_f1=0.6917, val_auc=0.7761, best_iter=999 (156.5s)
XGB: depth=6, lr=0.1, col=0.9... val_f1=0.6972, val_auc=0.7815, best_iter=999 (150.6s)
XGB: depth=8, lr=0.1, col=0.7... val_f1=0.6979, val_auc=0.7821, best_iter=599 (111.5s)

                    config  best_iter  val_acc   val_f1  val_auc     time_s
 depth=4, lr=0.1, col=0.7        999 0.700223 0.687890 0.770984  92.993296
 depth=6, lr=0.1, col=0.7        999 0.708218 0.697049 0.780914 132.337246
depth=6, lr=0.05, col=0.7        999 0.704194 0.691657 0.776135 156.533634
 depth=6, lr=0.1, col=0.9        999 0.708396 0.697178 0.781466 150.638489
 depth=8, lr=0.1, col=0.7        599 0.708582 0.697943 0.782130 111.505643


In [25]:
best_rf_cfg = rf_results_df.loc[rf_results_df['val_f1'].idxmax()]
print(f'Best RF config: {best_rf_cfg["config"]}')

# find original config dict to get params back
best_rf_params = rf_configs[rf_results_df['val_f1'].idxmax()]

print('\nRetraining best Random Forest...')
best_rf = RandomForestClassifier(
    n_estimators=best_rf_params['n_estimators'],
    max_depth=best_rf_params['max_depth'],
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=SEED,
    n_jobs=-1,
)
best_rf.fit(X_train_sparse, y_train)

rf_test_pred, rf_test_prob = evaluate(best_rf, X_test_sparse, y_test, 'RF Test')

Best RF config: n=300, depth=None

Retraining best Random Forest...

=== RF Test Results ===
Accuracy:  0.7127
Precision: 0.7269
Recall:    0.6814
F1 Score:  0.7034
ROC AUC:   0.7827

Classification Report:
               precision    recall  f1-score   support

Not Sarcastic       0.70      0.74      0.72    141666
    Sarcastic       0.73      0.68      0.70    141666

     accuracy                           0.71    283332
    macro avg       0.71      0.71      0.71    283332
 weighted avg       0.71      0.71      0.71    283332



In [26]:
best_xgb_cfg = xgb_results_df.loc[xgb_results_df['val_f1'].idxmax()]
print(f'Best XGB config: {best_xgb_cfg["config"]}')

# parse best config back
best_cfg = xgb_configs[xgb_results_df['val_f1'].idxmax()]
best_xgb = xgb.XGBClassifier(
    n_estimators=best_xgb_cfg['best_iter'] + 1,  # use best iteration directly
    max_depth=best_cfg['max_depth'],
    learning_rate=best_cfg['learning_rate'],
    colsample_bytree=best_cfg['colsample_bytree'],
    subsample=0.8,
    min_child_weight=5,
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    verbosity=0,
)
best_xgb.fit(X_train_dense, y_train)
xgb_test_pred, xgb_test_prob = evaluate(best_xgb, X_test_dense, y_test, 'XGBoost Test (SVD)')

# Best RF unchanged — retrain on sparse
best_rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=5,
    max_features='sqrt', random_state=SEED, n_jobs=-1,
)
best_rf.fit(X_train_sparse, y_train)
rf_test_pred, rf_test_prob = evaluate(best_rf, X_test_sparse, y_test, 'RF Test')

Best XGB config: depth=8, lr=0.1, col=0.7

=== XGBoost Test (SVD) Results ===
Accuracy:  0.7074
Precision: 0.7229
Recall:    0.6725
F1 Score:  0.6968
ROC AUC:   0.7814

Classification Report:
               precision    recall  f1-score   support

Not Sarcastic       0.69      0.74      0.72    141666
    Sarcastic       0.72      0.67      0.70    141666

     accuracy                           0.71    283332
    macro avg       0.71      0.71      0.71    283332
 weighted avg       0.71      0.71      0.71    283332


=== RF Test Results ===
Accuracy:  0.7127
Precision: 0.7269
Recall:    0.6814
F1 Score:  0.7034
ROC AUC:   0.7827

Classification Report:
               precision    recall  f1-score   support

Not Sarcastic       0.70      0.74      0.72    141666
    Sarcastic       0.73      0.68      0.70    141666

     accuracy                           0.71    283332
    macro avg       0.71      0.71      0.71    283332
 weighted avg       0.71      0.71      0.71    283332



In [27]:
dump(best_rf, 'random_forest_model.joblib')
dump(best_xgb, 'xgboost_model.joblib')
dump(tfidf, 'tfidf_vectorizer_tree.joblib')
dump(sub_stats, 'subreddit_stats.joblib')
print('Models saved.')

Models saved.
